In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
import os
import sys


In [0]:
current_dir = os.getcwd()

sys.path.append(current_dir)

## **Customer**

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customer")

In [0]:
df_cust = df_cust.withColumn("domain", split(col('email') , '@' )[1])

In [0]:
df_cust = df_cust.withColumn("phone_number" , regexp_replace("phone_number" , r"[^0-9]" , ""))

In [0]:
df_cust = df_cust.withColumn("full_name" , concat_ws(" ", col("first_name"), col("last_name")))
df_cust = df_cust.drop("first_name", "last_name")

In [0]:
from custom_utils import transformations

In [0]:
cust_obj = transformations()

cust_df_trans = cust_obj.dedup(df_cust , ['customer_id'],'last_updated_timestamp')

In [0]:
df_cust = cust_obj.process_timestamp(cust_df_trans)
display(df_cust)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):

    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")
else:
    cust_obj.upsert(spark , df_cust, ['customer_id'], 'customers', 'last_updated_timestamp')

In [0]:
df = spark.read.table("pysparkdbt.silver.customers")
display(df)
df.count()

## Drivers table


In [0]:
df_driver = spark.read.table("pysparkdbt.bronze.drivers")
display(df_driver)

In [0]:
df_driver = df_driver.withColumn("phone_number" , regexp_replace("phone_number" , r"[^0-9]" , ""))

In [0]:
df_driver = df_driver.withColumn("full_name" , concat_ws(" ", col("first_name"), col("last_name")))
df_driver = df_driver.drop("first_name", "last_name")

In [0]:
driver_obj = transformations()

In [0]:
df_driver = driver_obj.dedup(df_driver , ['driver_id'],'last_updated_timestamp')


In [0]:
df_driver = driver_obj.process_timestamp(df_driver)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):

    df_driver.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.drivers")
else:
    driver_obj.upsert(spark , df_driver, ['driver_id'],'drivers' , 'last_updated_timestamp')

In [0]:
df = spark.read.table("pysparkdbt.silver.drivers")
display(df)
df.count()

## locations table 


In [0]:
df_loc = spark.read.table("pysparkdbt.bronze.location")


In [0]:
loc_obj = transformations()


In [0]:
df_loc = loc_obj.dedup(df_loc , ['location_id'],'last_updated_timestamp')
df_loc = loc_obj.process_timestamp(df_loc)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.location"):

    df_loc.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.location")
else:
    loc_obj.upsert(spark , df_loc, ['location_id'],'location' , 'last_updated_timestamp')


In [0]:
df_loc = spark.read.table("pysparkdbt.silver.location")
display(df_loc)

### Payments table

In [0]:
df_pay = spark.read.table("pysparkdbt.bronze.payment")
display(df_pay)

In [0]:
df_pay = df_pay.withColumn("online_payment_status" ,
                           when(((   col('payment_method') == 'Card') & (col('payment_status') == 'Sucess')) , "online-Sucess")
                           .when(((   col('payment_method') == 'Card') & (col('payment_status') == 'Failed')) , "online-Sucess")
                           .when(((   col('payment_method') == 'Card') & (col('payment_status') == 'Pending')) , "online-Sucess")
                           .otherwise("offline")
                            )


In [0]:
payment_obj = transformations()
df_pay = payment_obj.dedup(df_pay , ['payment_id'],'last_updated_timestamp')
df_pay = payment_obj.process_timestamp(df_pay)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payment"):

    df_pay.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payment")
else:
    payment_obj.upsert(spark , df_pay, ['payment_id'],'payment' , 'last_updated_timestamp')

### VICHELES

In [0]:
df_veh = spark.read.table("pysparkdbt.bronze.vehicles")
display(df_veh)

In [0]:
df_veh = df_veh.withColumn("make" , upper(col("make")))

In [0]:
veh_obj = transformations()
df_veh = veh_obj.dedup(df_veh , ['vehicle_id'],'last_updated_timestamp')
df_veh = veh_obj.process_timestamp(df_veh)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicle"):

    df_veh.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.vehicle")
else:
    veh_obj.upsert(spark , df_veh, ['vehicle_id'],'vehicle' , 'last_updated_timestamp')

### trips

In [0]:
df_trip = spark.read.table("pysparkdbt.bronze.trips")



In [0]:
df_trip = df_trip.drop('payment_method')



In [0]:
df_tr = transformations()
df_trip = df_tr.process_timestamp(df_trip)
df_trip = df_tr.dedup(df_trip , ['trip_id'],'last_updated_timestamp')

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.trip"):

    df_trip.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.trip")
else:
    df_tr.upsert(spark , df_trip, ['trip_id'],'trip' , 'last_updated_timestamp')

In [0]:
df = spark.read.table("pysparkdbt.silver.trip")
display(df)